# Duka Electronics H1 Sales Exploration

This notebook explores Duka Electronics sales data for January to June 2026. The analysis examines branch performance, product performance, weekend versus weekday sales, monthly revenue and Premium online orders.

## Executive Summary

- Duka Electronics generated **KES 71,792,500** in total revenue from **1,000 orders** during the first half of 2026.
- Nairobi was the strongest branch, producing **KES 26,106,100** from **376 orders**.
- Laptops generated the most product revenue at **KES 41,535,000**, while Tablets sold the most units at **691 units**.
- April was the best revenue month at **KES 13,518,300**, which was **24.0% higher** than June, the weakest month at KES 10,903,400.
- Premium online orders from Nairobi and Mombasa totaled **83 orders** and generated **KES 10,218,000** in revenue.

In [1]:
%%writefile generate_data.py
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
n = 1000

products = {
    "Phone": 15000,
    "Laptop": 65000,
    "Headphones": 2500,
    "Tablet": 28000,
    "Charger": 1200
}

sales = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "order_date": pd.to_datetime("2026-01-01")
                  + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "branch": rng.choice(
        ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"],
        n,
        p=[0.4, 0.2, 0.15, 0.15, 0.1]
    ),
    "product": rng.choice(list(products), n),
    "quantity": rng.integers(1, 6, n),
    "channel": rng.choice(
        ["In-store", "Online", "Phone order"],
        n,
        p=[0.55, 0.35, 0.10]
    ),
})

sales["unit_price"] = sales["product"].map(products)

sales.to_csv("duka_sales.csv", index=False)

Overwriting generate_data.py


In [2]:
!python generate_data.py

## Task 1: First Five Minutes Inspection

I will load the sales data with the order date parsed correctly, then inspect the first and last rows, shape, data types, missing values, date range and key categories before doing any analysis.

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

print("First 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())

print("\nShape:")
print(df.shape)

print("\nData types and non-null values:")
df.info()

print("\nSummary statistics:")
display(df.describe())

print("\nMissing values:")
print(df.isna().sum())

print("\nDate range:")
print("Earliest:", df["order_date"].min())
print("Latest:", df["order_date"].max())

print("\nDistinct values:")
print(df.nunique())

print("\nBranch distribution:")
print(df["branch"].value_counts(normalize=True).round(3))

print("\nChannel distribution:")
print(df["channel"].value_counts(normalize=True).round(3))

First 5 rows:


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500



Last 5 rows:


,order_id,order_date,branch,product,quantity,channel,unit_price
995,1996,2026-03-04,Mombasa,Laptop,5,In-store,65000
996,1997,2026-03-06,Nairobi,Tablet,2,Phone order,28000
997,1998,2026-01-20,Nairobi,Laptop,3,Online,65000
998,1999,2026-03-02,Nairobi,Headphones,2,In-store,2500
999,2000,2026-01-30,Nakuru,Charger,4,In-store,1200



Shape:
(1000, 7)

Data types and non-null values:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[ns]
 2   branch      1000 non-null   object        
 3   product     1000 non-null   object        
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   object        
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 54.8+ KB

Summary statistics:


,order_id,order_date,quantity,unit_price
count,1000.000000,1000,1000.000000,1000.000000
mean,1500.500000,2026-03-30 08:44:09.600000,3.012000,23918.300000
min,1001.000000,2026-01-01 00:00:00,1.000000,1200.000000
25%,1250.750000,2026-02-14 00:00:00,2.000000,2500.000000
50%,1500.500000,2026-03-29 00:00:00,3.000000,15000.000000
75%,1750.250000,2026-05-14 00:00:00,4.000000,28000.000000
max,2000.000000,2026-06-29 00:00:00,5.000000,65000.000000
std,288.819436,NaN,1.416991,23908.790546



Missing values:
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64

Date range:
Earliest: 2026-01-01 00:00:00
Latest: 2026-06-29 00:00:00

Distinct values:
order_id      1000
order_date     179
branch           5
product          5
quantity         5
channel          3
unit_price       5
dtype: int64

Branch distribution:
branch
Nairobi    0.376
Mombasa    0.186
Nakuru     0.162
Kisumu     0.159
Eldoret    0.117
Name: proportion, dtype: float64

Channel distribution:
channel
In-store       0.579
Online         0.320
Phone order    0.101
Name: proportion, dtype: float64


### Inspection Findings

The dataset contains **1,000 rows and 7 columns**. There are **no missing values** in any column, and `order_date` was correctly loaded as a datetime column. The data covers the period from **1 January 2026 to 29 June 2026**. There are **5 branches, 5 products and 3 sales channels** in the dataset. Nairobi has the largest share of orders at **37.6%**, while In-store is the most common sales channel at **57.9%**.

## Task 2: Create Derived Columns

I will create the revenue, month, weekday, is_weekend and price_band columns. These columns will make it easier to compare sales performance across time, products and order types.

In [4]:
df["revenue"] = df["quantity"] * df["unit_price"]

df["month"] = df["order_date"].dt.month_name()

df["weekday"] = df["order_date"].dt.day_name()

df["is_weekend"] = df["weekday"].isin(["Saturday", "Sunday"])

df["price_band"] = np.where(
    df["unit_price"] >= 28000,
    "Premium",
    "Value"
)

display(
    df[
        [
            "order_id",
            "order_date",
            "branch",
            "product",
            "quantity",
            "unit_price",
            "revenue",
            "month",
            "weekday",
            "is_weekend",
            "price_band"
        ]
    ].head()
)

,order_id,order_date,branch,product,quantity,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Nakuru,Phone,3,15000,45000,June,Wednesday,False,Value
1,1002,2026-02-02,Nairobi,Laptop,4,65000,260000,February,Monday,False,Premium
2,1003,2026-01-05,Nakuru,Tablet,5,28000,140000,January,Monday,False,Premium
3,1004,2026-04-26,Nairobi,Headphones,5,2500,12500,April,Sunday,True,Value
4,1005,2026-03-07,Nairobi,Headphones,3,2500,7500,March,Saturday,True,Value


## Task 3: Branch Performance

I will rank each branch by total revenue and by number of orders, then compare the two rankings to see whether the same branches lead on both measures.

In [5]:
branch_revenue = (
    df.groupby("branch")["revenue"]
      .sum()
      .sort_values(ascending=False)
)

branch_orders = (
    df["branch"]
      .value_counts()
)

print("Branches ranked by total revenue:")
print(branch_revenue)

print("\nBranches ranked by number of orders:")
print(branch_orders)

Branches ranked by total revenue:
branch
Nairobi    26106100
Mombasa    14003700
Nakuru     11970400
Kisumu     10868100
Eldoret     8844200
Name: revenue, dtype: int64

Branches ranked by number of orders:
branch
Nairobi    376
Mombasa    186
Nakuru     162
Kisumu     159
Eldoret    117
Name: count, dtype: int64


### Task 3 Answer

The branch rankings are the same for both total revenue and number of orders. Nairobi ranks first with **KES 26,106,100** from **376 orders**, followed by Mombasa, Nakuru, Kisumu and Eldoret. This suggests that branches with more orders also generated more total revenue in this dataset. Nairobi's lead is especially strong because it has far more orders than the other branches.

## Task 4: Product Performance

I will identify the top three products by total revenue and by total units sold. I will then compare the two lists and explain why they may differ.

In [6]:
top_products_revenue = (
    df.groupby("product")["revenue"]
      .sum()
      .nlargest(3)
)

top_products_units = (
    df.groupby("product")["quantity"]
      .sum()
      .nlargest(3)
)

print("Top 3 products by revenue:")
print(top_products_revenue)

print("\nTop 3 products by units sold:")
print(top_products_units)

Top 3 products by revenue:
product
Laptop    41535000
Tablet    19348000
Phone      8880000
Name: revenue, dtype: int64

Top 3 products by units sold:
product
Tablet    691
Laptop    639
Phone     592
Name: quantity, dtype: int64


### Task 4 Answer

The top three products by revenue are Laptop at **KES 41,535,000**, Tablet at **KES 19,348,000** and Phone at **KES 8,880,000**. The top three products by units sold are Tablet with **691 units**, Laptop with **639 units** and Phone with **592 units**. The rankings are different because revenue depends on both quantity sold and unit price. Laptops generate the most revenue even though Tablets sold more units because Laptops have a much higher unit price.

## Task 5: Weekend vs Weekday Performance

I will compare weekend and weekday orders using the number of orders, total revenue and average order revenue.

In [7]:
weekend_summary = (
    df.groupby("is_weekend")["revenue"]
      .agg(["count", "sum", "mean"])
)

weekend_summary.index = ["Weekday", "Weekend"]

print(weekend_summary)

         count       sum          mean
Weekday    723  51856200  71723.651452
Weekend    277  19936300  71972.202166


### Task 5 Answer

Weekdays had **723 orders** and generated **KES 51,856,200** in total revenue, while weekends had **277 orders** and generated **KES 19,936,300**. The average order revenue was very similar: about **KES 71,724** on weekdays and **KES 71,972** on weekends. This means weekends had fewer orders, but the typical weekend order was slightly higher in value. The main reason weekday revenue is much higher is the larger number of weekday orders.

## Task 6: Best and Worst Revenue Month

I will calculate total revenue for each month, identify the best and worst months, and calculate the percentage difference between them.

In [8]:
monthly_revenue = (
    df.groupby("month")["revenue"]
      .sum()
)

month_order = [
    "January", "February", "March",
    "April", "May", "June"
]

monthly_revenue = monthly_revenue.reindex(month_order)

best_month = monthly_revenue.idxmax()
worst_month = monthly_revenue.idxmin()

best_revenue = monthly_revenue.max()
worst_revenue = monthly_revenue.min()

percentage_difference = (
    (best_revenue - worst_revenue) / worst_revenue * 100
)

print(monthly_revenue)
print(f"\nBest month: {best_month} - KES {best_revenue:,}")
print(f"Worst month: {worst_month} - KES {worst_revenue:,}")
print(f"Percentage difference: {percentage_difference:.1f}%")

month
January     11786600
February    11772000
March       12396800
April       13518300
May         11415400
June        10903400
Name: revenue, dtype: int64

Best month: April - KES 13,518,300
Worst month: June - KES 10,903,400
Percentage difference: 24.0%


### Task 6 Answer

April was the best month for revenue at **KES 13,518,300**, while June was the worst month at **KES 10,903,400**. April's revenue was **24.0% higher** than June's revenue. This shows that sales performance varied noticeably across the first half of 2026.

## Task 7: Premium Online Orders in Nairobi and Mombasa

I will filter Premium online orders from Nairobi and Mombasa, count the matching orders, calculate their total revenue and export the results to the required CSV file.

In [9]:
premium_online = df[
    (df["price_band"] == "Premium")
    & (df["channel"] == "Online")
    & (df["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

premium_count = len(premium_online)
premium_revenue = premium_online["revenue"].sum()

print(f"Number of Premium online orders: {premium_count}")
print(f"Total revenue: KES {premium_revenue:,}")

import os
os.makedirs("outputs", exist_ok=True)

premium_online.to_csv(
    "outputs/premium_online_coast_capital.csv",
    index=False
)

print("CSV exported successfully.")

Number of Premium online orders: 83
Total revenue: KES 10,218,000
CSV exported successfully.


### Task 7 Answer

There are **83 Premium online orders** from Nairobi and Mombasa. Together, these orders generated **KES 10,218,000** in revenue. The filtered orders were exported to `outputs/premium_online_coast_capital.csv` with the index excluded.

## Task 8: Spreadsheet Validation

I reproduced the branch revenue ranking in Google Sheets using SUMIFS, added a region column using XLOOKUP and created a PivotTable of revenue by branch and channel. The spreadsheet grand total was **KES 71,792,500**, which matches the total revenue calculated in pandas. This confirms that the spreadsheet and notebook results are consistent.

In [10]:
%%writefile requirements.txt
pandas
numpy

Overwriting requirements.txt
